# Raw Data Exploration and Profiling

## 1. Purpose and Scope

This notebook explores the raw eBay Browse API data landed by DLTHub in Google Cloud Storage. The goal is to understand source structure, volume, completeness, identifier behavior, and relationships between discovery results and enriched item details before designing Bronze and Silver transformations.

**Scope**
- Profile the `browse_search` and `item_details` raw resources.
- Preserve the source representation and DLTHub technical metadata during exploration.
- Record observed behavior and limitations to inform downstream data modeling.

## 2. Data Loading

The following cells define the raw GCS location and load the two business-facing resources into Spark DataFrames. The raw files are read in place; this notebook does not alter them.


In [0]:
RAW_BUCKET = "gs://market-intelligence-raw"
print(RAW_BUCKET)

In [0]:
item_details_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/item_details/*.jsonl.gz"
)

display(item_details_df.limit(5))

In [0]:
browse_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/browse_search/*.jsonl.gz"
)

display(browse_df.limit(5))

## 3. Profiling Setup and Dataset Inventory

Define reusable references for the loaded DataFrames and establish baseline row and column counts. These measurements describe the data available at the time this notebook is run.


In [0]:
from pyspark.sql import functions as F

# ============================================================
# Dataset references
# ============================================================

DATASETS = {
    "browse_search": browse_df,
    "item_details": item_details_df,
}

for name, df in DATASETS.items():
    print(f"{name}:")
    print(f"  rows    : {df.count():,}")
    print(f"  columns : {len(df.columns):,}")
    print()

## 4. Schema Profiling

Inspect column names, Spark data types, and nullability for each source resource. This is a structural inventory, not a business-level type-casting or schema standardization step.


In [0]:
def profile_schema(df, dataset_name: str):
    rows = []

    for field in df.schema.fields:
        rows.append(
            (
                field.name,
                field.dataType.simpleString(),
                field.nullable,
            )
        )

    return spark.createDataFrame(
        rows,
        ["column_name", "data_type", "nullable"]
    ).withColumn(
        "dataset",
        F.lit(dataset_name)
    ).select(
        "dataset",
        "column_name",
        "data_type",
        "nullable",
    )

In [0]:
browse_schema = profile_schema(
    browse_df,
    "browse_search"
)

display(browse_schema)

In [0]:
item_details_schema = profile_schema(
    item_details_df,
    "item_details"
)

display(item_details_schema)

## 5. Completeness Profiling

Measure non-null and null values by column for both datasets. Nulls are treated as profiling signals; their meaning may depend on listing type, category, or API response behavior and should not automatically be classified as defects.


In [0]:
def profile_completeness(df):
    total_rows = df.count()

    expressions = []

    for column_name in df.columns:
        expressions.extend([
            F.count(F.col(column_name)).alias(f"{column_name}__non_null"),
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{column_name}__null"),
        ])

    result = df.agg(*expressions)

    profile_rows = []

    for column_name in df.columns:
        non_null_col = f"{column_name}__non_null"
        null_col = f"{column_name}__null"

        row = result.select(
            F.lit(column_name).alias("column_name"),
            F.col(non_null_col).alias("non_null_count"),
            F.col(null_col).alias("null_count"),
        ).withColumn(
            "total_rows",
            F.lit(total_rows)
        ).withColumn(
            "null_percentage",
            F.round(
                F.col("null_count") / F.col("total_rows") * 100,
                2
            )
        )

        profile_rows.append(row)

    final_df = profile_rows[0]

    for row in profile_rows[1:]:
        final_df = final_df.unionByName(row)

    return final_df.orderBy(
        F.col("null_percentage").desc()
    )

In [0]:
browse_completeness = profile_completeness(browse_df)

display(browse_completeness)

In [0]:
item_details_completeness = profile_completeness(item_details_df)

display(item_details_completeness)

## 6. Cardinality, Repetition, and Resource Coverage

Assess `item_id` cardinality, identify repeated IDs, inspect their frequency, and compare the distinct listing IDs present in Browse discovery versus item details.


In [0]:
from pyspark.sql import functions as F

# ============================================================
# Item ID cardinality
# ============================================================

for name, df in DATASETS.items():

    total_rows = df.count()

    distinct_items = (
        df.select("item_id")
        .where(F.col("item_id").isNotNull())
        .distinct()
        .count()
    )

    duplicate_rows = total_rows - distinct_items

    print(f"\n{name}")
    print("-" * 50)
    print(f"Total rows           : {total_rows:,}")
    print(f"Distinct item_ids    : {distinct_items:,}")
    print(f"Duplicate occurrences: {duplicate_rows:,}")

In [0]:
browse_item_frequency = (
    browse_df
    .groupBy("item_id")
    .count()
    .orderBy(F.col("count").desc())
)

display(browse_item_frequency.limit(20))

In [0]:
item_details_item_frequency = (
    item_details_df
    .groupBy("item_id")
    .count()
    .orderBy(F.col("count").desc())
)

display(item_details_item_frequency.limit(20))

In [0]:
repeated_items = (
    browse_df
    .groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

browse_duplicates = (
    browse_df
    .join(repeated_items, on="item_id", how="inner")
    .orderBy("item_id")
)

display(
    browse_duplicates.select(
        "item_id",
        "title",
        "price__value",
        "price__currency",
        "seller__username",
        "condition",
        "_dlt_load_id"
    ).limit(100)
)

In [0]:
browse_unique_items = (
    browse_df
    .select("item_id")
    .where(F.col("item_id").isNotNull())
    .distinct()
    .withColumn("in_browse", F.lit(True))
)

details_unique_items = (
    item_details_df
    .select("item_id")
    .where(F.col("item_id").isNotNull())
    .distinct()
    .withColumn("in_details", F.lit(True))
)

coverage = (
    browse_unique_items
    .join(details_unique_items, on="item_id", how="left")
    .fillna({"in_details": False})
)

display(
    coverage.groupBy("in_details")
    .count()
)

## 7. Repeated Browse Record Investigation

This section investigates the grain and variation of repeated `item_id` observations in `browse_search`. Repeated IDs are not removed: the purpose is to understand their origin and business-field behavior before making downstream modeling decisions.


In [0]:
from pyspark.sql import functions as F

# Identify item IDs that occur more than once
repeated_ids = (
    browse_df
    .groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

# Retrieve every occurrence of those items
repeated_records = browse_df.join(
    repeated_ids,
    on="item_id",
    how="inner"
)

# Compare distinct values across selected business columns
comparison_columns = [
    "title",
    "price__value",
    "price__currency",
    "seller__username",
    "condition",
    "condition_id",
]

duplicate_profile = (
    repeated_records
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        *[
            F.countDistinct(F.col(c)).alias(f"{c}_distinct")
            for c in comparison_columns
        ]
    )
)

display(
    duplicate_profile
    .orderBy(F.col("occurrences").desc())
    .limit(50)
)

In [0]:
from pyspark.sql import functions as F

(
    browse_df
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct("_dlt_load_id").alias("distinct_loads"),
        F.countDistinct("title").alias("distinct_titles"),
        F.countDistinct("price__value").alias("distinct_prices")
    )
    .filter(F.col("occurrences") > 1)
    .groupBy("distinct_loads")
    .count()
    .orderBy("distinct_loads")
    .show()
)

### 7.1 Duplicate Record and Business-Field Variation Analysis

### Objective

Investigate repeated `item_id` values in the `browse_search` dataset to determine whether they represent identical business records or multiple distinct observations of the same listing.

A repeated item identifier does not necessarily indicate a data quality issue. The same eBay listing may appear in multiple search results, and its returned attributes may differ between observations.

### Methodology

The analysis is performed at the `item_id` grain:

1. Identify item IDs occurring more than once.
2. Count distinct DLTHub load IDs for each repeated item to investigate whether repetitions occur within or across ingestion loads.
3. Exclude technical metadata (`_dlt_id`, `_dlt_load_id`) and the grouping key (`item_id`) when comparing business records.
4. Count distinct business records for each repeated item ID.

This analysis is descriptive. It does not remove records or modify the raw data.

### Observations

The profiling results returned 9,772 repeated item ID groups.

| Distinct business records per item | Number of item ID groups |
| ---------------------------------: | -----------------------: |
|                                  1 |                    5,111 |
|                                  2 |                    4,368 |
|                                  3 |                      268 |
|                                  4 |                       24 |
|                                  5 |                        1 |

All 9,772 groups in the load-level analysis had one distinct `_dlt_load_id`, indicating that the repeated observations occurred within a single ingestion load.

Of the repeated item ID groups, 5,111 had identical values across the compared business columns, while 4,661 had more than one distinct business record.

### Interpretation

The results establish that repeated item IDs cannot automatically be treated as exact duplicates.

The observations are consistent with overlapping search-query results, but the precise cause of the repetitions has not yet been established.

Differences between business records could involve price, condition, seller attributes, listing metadata, or other returned fields. The specific fields responsible for the variation remain to be investigated.

### Limitations and Next Steps

* The analysis covers the currently available raw dataset and does not establish permanent source-system behavior.
* A distinct business record does not necessarily represent a price change or a chronological listing update.
* The analysis has not yet established whether differences originate from search-query overlap, API response context, or listing changes.

**Next:** Profile field-level variation across repeated item IDs to identify which business attributes differ. Inspect representative records before defining the Silver-layer deduplication and business-grain strategy.

The raw Bronze input will remain unchanged. Any future deduplication or record-selection logic will be implemented in the appropriate downstream transformation layer.


In [0]:
from pyspark.sql import functions as F

business_columns = [
    c for c in browse_df.columns
    if c not in ["_dlt_id", "_dlt_load_id"]
]

record_variation = (
    browse_df
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in business_columns])
        ).alias("distinct_business_records")
    )
    .filter(F.col("occurrences") > 1)
)

record_variation.groupBy("distinct_business_records") \
    .count() \
    .orderBy("distinct_business_records") \
    .show()

In [0]:
from pyspark.sql import functions as F

business_columns = [
    c for c in browse_df.columns
    if c not in ["_dlt_id", "_dlt_load_id", "item_id"]
]

repeated_ids = (
    browse_df.groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

repeated_records = browse_df.join(
    repeated_ids,
    on="item_id",
    how="inner"
)

variation_expressions = [
    F.countDistinct(F.col(c)).alias(c)
    for c in business_columns
]

variation_profile = repeated_records.groupBy("item_id").agg(
    F.count("*").alias("occurrences"),
    *variation_expressions
)

varying_fields = [
    F.sum(
        F.when(F.col(c) > 1, 1).otherwise(0)
    ).alias(c)
    for c in business_columns
]

field_variation_summary = variation_profile.agg(
    *varying_fields
)

display(field_variation_summary)

### 7.2 Field-Level Variation Across Repeated Item IDs

#### Objective

Identify the business attributes responsible for distinct records among repeated `item_id` groups in `browse_search`.

The previous analysis established that repeated item IDs can contain multiple distinct business records. This analysis determines which individual fields contribute to those differences.

#### Results

The field-level variation profile identified the following business columns with non-zero variation:

| Business column               | Repeated item ID groups with variation |
| ----------------------------- | -------------------------------------: |
| `item_web_url`                |                                  4,625 |
| `priority_listing`            |                                    294 |
| `seller__feedback_score`      |                                     34 |
| `top_rated_buying_experience` |                                     14 |
| `price__value`                |                                      8 |
| `seller__feedback_percentage` |                                      1 |

All other business columns included in the profile had zero variation.

The counts represent the number of repeated item ID groups containing more than one distinct value for the respective field.

#### Interpretation

Variation among repeated records is concentrated in a limited number of business attributes.

The `item_web_url` field accounts for the largest number of varying groups. Seller feedback attributes and listing flags also differ for some repeated item IDs.

Only eight repeated item ID groups exhibit variation in `price__value`. Therefore, repeated discovery records should not automatically be interpreted as price changes.

The observed differences establish that some repeated records are not exact business-record duplicates. However, the analysis does not establish whether the differences originate from overlapping search queries, API response context, or changes to listing attributes.

#### Limitations

* The profile does not establish the chronological order of observations.
* It does not identify the specific values responsible for variation.
* It does not establish whether a difference is meaningful for downstream business analytics.
* The findings describe the current raw dataset and should not be treated as permanent source-system guarantees.

#### Next Step

Inspect representative repeated item IDs and compare their actual field values, particularly `item_web_url`, `priority_listing`, seller feedback attributes, and `price__value`.

This investigation will inform the downstream Silver-layer record-grain and deduplication strategy. No records are removed or modified during raw data profiling.



In [0]:
from pyspark.sql import functions as F

varying_item_ids = (
    repeated_records
    .groupBy("item_id")
    .agg(
        F.countDistinct("item_web_url").alias("url_variants"),
        F.countDistinct("price__value").alias("price_variants"),
        F.countDistinct("priority_listing").alias("priority_variants")
    )
    .filter(
        (F.col("url_variants") > 1) |
        (F.col("price_variants") > 1) |
        (F.col("priority_variants") > 1)
    )
    .select("item_id")
)

display(
    repeated_records
    .join(varying_item_ids, "item_id")
    .select(
        "item_id",
        "title",
        "item_web_url",
        "price__value",
        "price__currency",
        "priority_listing",
        "seller__feedback_score",
        "seller__feedback_percentage",
        "top_rated_buying_experience",
        "_dlt_load_id"
    )
    .orderBy("item_id")
    .limit(100)
)

### 7.3 Inspection of Representative Repeated Records

#### Objective

Inspect actual business-field values for repeated item IDs to understand the differences identified during field-level variation profiling.

#### Observations

Representative records show the same eBay listing appearing under different search queries.

For example, the same Xbox listing appears in searches associated with SSDs, Xbox consoles, and gaming consoles. The listing retains the same item ID, title, and price, while the `item_web_url` contains different search-related query parameters.

Similar patterns were observed for keyboard, tablet, and other product listings.

#### Interpretation

The inspected records provide evidence that overlapping search queries contribute to repeated Browse observations.

The `item_web_url` field contains search-context parameters, including `_skw`, which can differ even when the underlying listing is unchanged.

This indicates that `item_id` represents listing identity, whereas each Browse result represents a discovery observation.

The two concepts should not be treated as equivalent when designing downstream data models.

#### Limitations

The inspected records are representative samples, not an exhaustive analysis of every repeated item ID.

The examples establish search-context variation for the inspected records but do not explain every instance of business-field variation identified in the full dataset.

Differences in price, seller feedback, and listing flags require separate investigation before determining whether they represent meaningful listing changes.

#### Next Step

Investigate the remaining varying business attributes and quantify how frequently they differ independently of search-context URL parameters.

Use the findings to define the listing grain and observation-handling strategy for the Silver layer, while preserving the raw source observations.
